In [1]:
import polars as pl
import os
from pathlib import Path

def process_snow_water_2003_12():
    """
    Process snow water data specifically for December 2003.
    Returns averaged values for each X,Y coordinate.
    """
    current_dir = os.getcwd()
    
    # Get all CSV files
    all_files = list(Path(current_dir).glob("us_ssmv11034tS__T0001TTNATS*.csv"))
    
    # Filter for 2003/12 files based on filename
    target_files = [
        f for f in all_files 
        if f.name[27:31] == '2003' and f.name[31:33] == '12'
    ]
    
    print(f"Found {len(target_files)} files for December 2003")
    
    # Initialize list to store valid dataframes
    valid_dfs = []
    
    # Process each file
    for file_path in target_files:
        try:
            if file_path.stat().st_size == 0:
                print(f"Skipping empty file: {file_path.name}")
                continue
                
            df = pl.read_csv(
                file_path,
                has_header=True,
                new_columns=['X', 'Y', 'Value', 'Year', 'Month']
            )
            
            # Replace invalid values with None
            df = df.with_columns(
                pl.when(pl.col('Value').is_in([55537, -9999]))
                .then(None)
                .otherwise(pl.col('Value'))
                .alias('Value')
            )
            
            valid_dfs.append(df)
            print(f"Processed {file_path.name}")
            
        except Exception as e:
            print(f"Error processing {file_path.name}: {e}")
            continue
    
    if not valid_dfs:
        print("No valid data found")
        return None
    
    # Combine all dataframes and compute averages
    result = (pl.concat(valid_dfs)
        .group_by(['X', 'Y'])
        .agg(
            pl.col('Value').mean().alias('Value'),
            pl.lit(2003).alias('Year'),
            pl.lit(12).alias('Month')
        )
    )
    
    # Write output
    output_file = "snow_water_equiv_2003_12_avg.csv"
    result.write_csv(output_file)
    print(f"Created {output_file}")
    
    return result

if __name__ == "__main__":
    result = process_snow_water_2003_12()

Found 23 files for December 2003
Processed us_ssmv11034tS__T0001TTNATS2003121105HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003120205HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003121505HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003121905HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003122605HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003123105HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003122805HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003121205HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003120305HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003122705HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003122005HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003121605HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003121305HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003122405HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003122905HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003121805HP001.csv
Processed us_ssmv11034tS__T0001TTNATS2003122105HP001.cs